#### This code returns the DATA needed to plot the histograms

In [1]:
from matplotlib.image import NonUniformImage
import matplotlib.pyplot as plt

## Functions

In [2]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method consist of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCA2(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    indexA=zx2['Index']
    
    #Empty Dataframes
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,


def PCA2(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    #print(X)
    check_for_nan_1 = X['Bx'].isnull()
    check_for_nan_2 = X['By'].isnull()
    check_for_nan_3 = X['Bz'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True or check_for_nan_2[i]==True or check_for_nan_3[i]==True :
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #print(data_copy)
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        covariance_matrix=data_copy.cov()
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#---------------------------------------------------------------------------------------------------------------

## Folders

In [3]:
#mag_folder="C:/Users/Ariel/Desktop/ESA/Data_Rosetta/2014_months/11"
#mag_folder="C:/Users/Ariel/Desktop/ESA/Data_Rosetta/2015_months/4"
mag_folder="C:/Users/Ariel/Desktop/ESA/Data_Rosetta/all_data"
lap_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/all_data"
#lap_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2014/noviembre"
#lap_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/abril"

In [4]:
#with np.printoptions(threshold=np.inf):
     #print(H.T)

In [5]:
#-------------------------------------------------------------------------------------------
outgassing="C:/Users/Ariel/Desktop/ESA/Data_GasProduction/gasproduction.txt"

title2=['Timestamp','Neutral gas density','Derived gas production rate']
path= str(outgassing)
data1= pd.read_table(path, header=None, names=title2, sep='\s+', engine='python', parse_dates=['Timestamp'])
 
    
#Filtering
outgassing_copy=data1.copy() 
for j in np.arange(0,len(outgassing_copy)):
    if outgassing_copy['Derived gas production rate'][j]<1000000: 
        outgassing_copy['Timestamp'][j]=np.nan

outgassing_copy=outgassing_copy.dropna()       
data_gas=outgassing_copy.reset_index(drop=True)
data_gas['date'] = pd.to_datetime(data_gas['Timestamp'])

<ipython-input-5-1a523961faef>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outgassing_copy['Timestamp'][j]=np.nan


KeyboardInterrupt: 

## CODE 2

## MAG and Gas data #2 using 1 second

In [ ]:
#mag_folder="C:/Users/Ariel/Desktop/ESA/Data_Rosetta/all_data"
#IMPORT DATA
directory= os.scandir(mag_folder)
list_of_files=get_directory(mag_folder)

newlist2=[]
for item in list_of_files:
    newlist2.append(mag_folder+'/'+str(item))
#print(newlist)

mag_table=[]

for j in np.arange(0,len(newlist2)):
#for j in np.arange(0,1):
    print(j)
    #Importing data
    titles=['Dates_and_Hours', '?', 'x', 'y', 'z', 'Bx', 'By', 'Bz', 'flag']
    data= pd.read_table(str(newlist2[j]), header=None, names=titles, sep='\s+', parse_dates=['Dates_and_Hours'])
    #print(data)
     #Filling the data gaps
    data.Dates_and_Hours = data.Dates_and_Hours.dt.round('1s') #round to one second for simplicity
    if data.shape[0] < 86400: # if the number of datapoints is lower than one day:
        #print('Data gaps detected, padding array....')
        data2 = data.rename(index=(data['Dates_and_Hours']-data.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
        data2 = data2.reindex(range(0,86400)) # now we just fill in the missing values
        newt = pd.date_range(start = data.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
        data2['Dates_and_Hours'] = newt # now fill in the times so there is no NaT
        data = data2 
        del(data2)
    elif data.shape[0] > 86400:
        error('Data file is too long, probably need to debug the code again....')
    #print('Done\n')
    
    
    #LOADING DATA
    #--------------------------------------------
    XX = data[['Dates_and_Hours','x','y','z','Bx', 'By', 'Bz']]
    
    #Calculates the magnetic field modulus for each point
    bmodulus=[]
    for i in np.arange(0,len(XX)):
        Bpoint=(XX['Bx'][i]**2+XX['By'][i]**2+XX['Bz'][i]**2)**(1/2)
        bmodulus.append(Bpoint)
    
    #Transform a list into a data column
    df1 = pd.DataFrame({'col':bmodulus})
    XX['bmodulus']=df1
    
    magneticfieldcolumn=df1['col']
    newdeltaB=newdeltas(magneticfieldcolumn)
    
    XX['deltas']=newdeltaB
    
    #Calculates the cometo-distance for each point
    distance=[]
    for i in np.arange(0,len(XX)):
        dpoint=(XX['x'][i]**2+XX['y'][i]**2+XX['z'][i]**2)**(1/2)
        distance.append(dpoint)
        
    #Transform a list into a data column
    df2 = pd.DataFrame({'col':distance})
    XX['cometo-centric distance']=df2
    XX2=XX[['Dates_and_Hours','x','y','z','bmodulus','deltas','cometo-centric distance']]
    
    mag_table.append(XX2)

In [ ]:
#-----------------------------------------
#lap_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/all_data"

#IMPORT DATA:
directory= os.scandir(lap_folder)
list_of_files=get_directory(lap_folder)

newlist=[]
for item in list_of_files:
    newlist.append(lap_folder+'/'+str(item))

#print(newlist)    
H1=[]
H2=[]

#ESTOY USANDO EL (1,2) Y EL (2,3)
#for i in np.arange(1,5): #For one day
for i in np.arange(0,len(newlist)): #For every day
        print(i)
        #Lap data
        #--------------------------------------------
        title2=['t_utc','?','npl','??','???','????']
        path= str(newlist[i])
        data1= pd.read_table(path, header=None, names=title2, sep=',', engine='python', parse_dates=['t_utc'])
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        data2=data2.reset_index(drop=True)
        data2=data2[['t_utc','npl']]
        #Resample
        data2['index'] = pd.to_datetime(data2['t_utc'])
        data2.set_index('index', inplace=True)
        data2=data2.resample('1S').mean()
        data2['t_utc'] = pd.to_datetime(data2.index.values)
        data2=data2.dropna() #Drop the nan values of the resample
        
        path_time= pd.to_datetime(data2['t_utc'])
        q=path_time.dt.year
        qq=path_time.dt.month
        qqq=path_time.dt.day
        qqqq=path_time.dt.hour
        qqqqq=path_time.dt.minute        
        
        #Chosing the specific MAG data
        for j in np.arange(0,len(mag_table)):
            if pd.to_datetime(mag_table[j]['Dates_and_Hours']).dt.year[0]==q[0] and pd.to_datetime(mag_table[j]['Dates_and_Hours']).dt.month[0]==qq[0] and pd.to_datetime(mag_table[j]['Dates_and_Hours']).dt.day[0]==qqq[0]:
                day_chosen=mag_table[j]
                break
        
        day_chosen_copy=day_chosen.copy()
        day_chosen_copy.set_index('Dates_and_Hours', inplace=True)      
        att = day_chosen_copy.reindex(data2['t_utc'], method='pad') 
        
        #-----------------------------
        #GAS SECTION

        # Chosing the specific GAS data
        filtered_df = data_gas.loc[(data_gas['date'].dt.year==q[0]) & (data_gas['date'].dt.month==qq[0]) & (data_gas['date'].dt.day==qqq[0])]
        
        'I know it is redundant to make a 1s resample and then a 60s resample instead of just a 60s resample, but it is made like that in order to begin the time'
        'interval at 00:00:00, it was the easiest and fastest way that I found'
        
        if len(filtered_df)!=0:
            #Filling the data gaps
            filtered_df.Timestamp = filtered_df.Timestamp.dt.round('1s') #round to one second for simplicity
            if filtered_df.shape[0] < 86400: # if the number of datapoints is lower than one day:
                    #print('Data gaps detected, padding array....')
                data3 = filtered_df.rename(index=(filtered_df['Timestamp']-filtered_df.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
                data3 = data3.reindex(range(0,86400)) # now we just fill in the missing values
                newt = pd.date_range(start = filtered_df.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
                data3['Timestamp'] = newt # now fill in the times so there is no NaT
                filtered_df = data3 
                del(data3)
            elif filtered_df.shape[0] > 86400:
                error('Data file is too long, probably need to debug the code again....')
                #print('Done\n')  

            #Resample
            filtered_df2=filtered_df.copy()
            filtered_df2['index'] = pd.to_datetime(filtered_df2['Timestamp'])
            filtered_df2.set_index('index', inplace=True)
            filtered_df2=filtered_df2.resample('60S').mean()
            filtered_df2['t_utc'] = pd.to_datetime(filtered_df2.index.values)
            
            #Here we have the 1Hz gas values 
            gas_values=filtered_df2['Derived gas production rate']
            #print('gas values')
            #print(gas_values)
            abc=[]
            for z in np.arange(0,len(gas_values)):
                #if there is a nan value it will be multiplied by a factor of 60
                lists=gas_values[z]*np.ones(60)
                abc=np.concatenate((abc, lists), axis=0)
            
            #Here we have gas data with 86400 seconds
            filtered_df['gas_values']=abc
            filtered_df.set_index('Timestamp', inplace=True)       

            att2 = filtered_df.reindex(data2['t_utc'], method='pad')   

            #Dataframe with gas production rates and magnetic data
            att['gas_production_rate']=att2['gas_values']
            final_dataframe=att[['bmodulus','cometo-centric distance','gas_production_rate']]
            
            import csv
            final_dataframe.to_csv('finaldataframe_'+str(i), index=False, sep='\t')    
#SACAR LO DEMAS            
            #print('Final dataframes')
            #print(final_dataframe)
            
            #plt.figure()
            #plt.plot(np.arange(0,len(final_dataframe)), final_dataframe['bmodulus'].astype(float))
            #plt.title('bmodulus')
            
            #HISTOGRAM FOR GAS/COMETO CENTRIC
            final_dataframe_copy=final_dataframe.copy()
            #Filter nan values        
            final_dataframe_copy=final_dataframe_copy.dropna(subset=['cometo-centric distance','gas_production_rate'])
            final_dataframe_copy=final_dataframe_copy.reset_index(drop=True)
            
            import math
            
            x=final_dataframe_copy['gas_production_rate']
            xx=x.astype(float)
            y=final_dataframe_copy['cometo-centric distance']
            yy=y.astype(float)
            xxx=[]
            
            
            for n in np.arange(0,len(xx)):
                xxx.append(math.log(xx[n],10))    

            
            #plt.figure()
            #plt.plot(xxx, yy)
            #plt.title('weas de cometo')
            
            
            #---------
            xmin=24.5
            xmax=29
            ymin=0
            ymax=500
            #---------
            H, xedges, yedges= np.histogram2d(xxx,yy, bins=100, range=[[xmin,xmax],[ymin,ymax]])
            H1.append(H)
            #np.savetxt("matriz2.txt", H)        
            
    
            #HISTOGRAM FOR GAS/BMODULUS
            final_dataframe_copy2=final_dataframe.copy()
            #Filter nan values        
            final_dataframe_copy2=final_dataframe_copy2.dropna(subset=['bmodulus','gas_production_rate'])
            final_dataframe_copy2=final_dataframe_copy2.reset_index(drop=True)
            
            x2=final_dataframe_copy2['gas_production_rate']
            xx2=x2.astype(float)
            y2=final_dataframe_copy2['bmodulus']
            yy2=y2.astype(float)
            xxx2=[]
            
            #---------
            ymin2=0
            ymax2=100
            #---------
            
            for m in np.arange(0,len(xx2)):
                xxx2.append(math.log(xx2[m],10))   
            
            plt.figure()
            plt.plot(xxx2,yy2)
            plt.title('bmodulus weas')
            
            HII, xedges2, yedges2= np.histogram2d(xxx2,yy2, bins=100, range=[[xmin,xmax],[ymin2,ymax2]])
            H2.append(HII)
            
            
            #----------------------------------------
            np.savetxt("matriz1_"+str(i) +".txt", H)
            np.savetxt("matriz2_"+str(i) +".txt", HII)
        
        else:
            H1.append(np.zeros(shape=(100,100)))
            H2.append(np.zeros(shape=(100,100)))
            
            #----------------------------------------
            np.savetxt("matriz1_"+str(i) +".txt", H)
            np.savetxt("matriz2_"+str(i) +".txt", HII)



In [ ]:
print(H)

In [ ]:
Done

### Matrix per month

In [ ]:
zeros = np.zeros(shape=(100,100))    
for i in np.arange(0,len(H1)):
    zeros=np.add(zeros,H1[i])

zeros2 = np.zeros(shape=(100,100))    
for j in np.arange(0,len(H2)):
    zeros2=np.add(zeros2,H2[j])    
    
print("Addition of two matrix") 
print(zeros)

print("Addition of two matrix") 
print(zeros2)


#np.savetxt("matrix1.txt", zeros)
#np.savetxt("matrix2.txt", zeros2)
#plt.plot(zeros2)

In [ ]:
print(zeros2)
plt.plot(zeros2)

## Plot #1 Cometo centric distance vs outgassing

In [ ]:
MM_prominence_data3="C:/Users/Ariel/Desktop/JUPYTER/Resultados/data_used_for_the_histograms/All_prominence_events3"

#Importing data
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmodulus', 'cometo-centric distance']
dataa= pd.read_table(str(MM_prominence_data3), header=None, names=titles, sep='\t')
data3=dataa.iloc[np.arange(1,len(dataa)),:] #delete the first row
data3=data3.reset_index(drop=True)


data3_copy=data3.copy()
#Filter nan values
for j in np.arange(0,len(data3)):
    if pd.isnull(data3_copy['gas_production_rates'][j])==True or pd.isnull(data3_copy['cometo-centric distance'][j])==True:
        data3_copy.drop(j, axis=0, inplace=True)        
data3_copy=data3_copy.reset_index(drop=True)

import math
x=data3_copy['gas_production_rates']
xx=x.astype(float)
y=data3_copy['cometo-centric distance']
yy=y.astype(float)


xxx=[]
for i in np.arange(0,len(xx)):
    #print(i)
    if np.isnan(xx[i])==True:
        xxx.append(np.nan)
    else:
        xxx.append(math.log(xx[i],10))

xmin=24.5
xmax=29
ymin=0
ymax=500
extent=[xmin,xmax,ymin,ymax]
HHH, xedges, yedges= np.histogram2d(xxx,yy, bins=100, range=[[xmin,xmax],[ymin,ymax]])

HHH_copy=HHH.copy()
for i in np.arange(0,len(HHH_copy[0])):
    for j in np.arange(0,len(HHH_copy[0])):
        if HHH_copy[i][j]==0:
            HHH_copy[i][j]=np.nan




plt.figure()
#plt.imshow(HHH.T ,origin='lower', extent=extent, aspect='auto')
plt.imshow(HHH_copy.T ,origin='lower', extent=extent, aspect='auto')
plt.colorbar(label='Amount of MM waves')
plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
plt.ylabel('r (km)')

In [ ]:
def matrix_division(array1,array2):
    array1_copy=array1.copy()
    for i in np.arange(0,len(array1)):
        for j in np.arange(0,len(array1[0])):
            if array2[i][j]==0:
                array1_copy[i][j]=0
            else:    
                array1_copy[i][j]=array1[i][j]/array2[i][j]
                
    return array1_copy   

### Normalized waves

In [ ]:
xedges=100
yedges=100
xmin=24.5
xmax=29
ymin=0
ymax=500
HHH, xedges, yedges= np.histogram2d(xxx,yy, bins=100, range=[[xmin,xmax],[ymin,ymax]])


H_cometo_centric=matrix_division(HHH,zeros)
matrix_copy=H_cometo_centric.copy()
for i in np.arange(0,len(matrix_copy[0])):
    for j in np.arange(0,len(matrix_copy[0])):
        if matrix_copy[i][j]==0:
            matrix_copy[i][j]=np.nan
            
maximum= H_cometo_centric.max()
print('el maximo es:')
print(maximum)
    
'CALCULANDO EL MINIMO DE LA MATRIZ LUEGO DE APLICAR NAN VALUES'
matrix_copy2=matrix_copy.copy()
for i in np.arange(0,len(matrix_copy2[0])):
    for j in np.arange(0,len(matrix_copy2[0])):
        if np.isnan(matrix_copy2[i][j])==True:
            matrix_copy2[i][j]=5000


minimum = matrix_copy2.min()
    
print('el minimo es:')    
print(minimum)      
            
#print(matrix_copy)            
extent = [24.5,29, 0, 500]
plt.figure()
#plt.imshow(H_cometo_centric.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(matrix_copy.T ,origin='lower', extent=extent, aspect='auto', cmap='coolwarm')
plt.imshow(matrix_copy.T ,origin='lower', extent=extent, aspect='auto')
plt.colorbar(label='Amount of normalized MM waves')
plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
plt.ylabel('r (km)')

### LAP data 

In [ ]:
plt.figure()
extent = [24.5,29, 0, 500]

zeros1_copy=zeros.copy()
for i in np.arange(0,len(zeros1_copy[0])):
    for j in np.arange(0,len(zeros1_copy[0])):
        if zeros1_copy[i][j]==0:
            zeros1_copy[i][j]=np.nan

maximum= zeros.max()
print('the maximum is:')
print(maximum)
    
'Calculating the minimum of the matrix after the nan values'
zeros1_copy2=zeros1_copy.copy()
for i in np.arange(0,len(zeros1_copy2[0])):
    for j in np.arange(0,len(zeros1_copy2[0])):
        if np.isnan(zeros1_copy2[i][j])==True:
            zeros1_copy2[i][j]=5000


minimum = zeros1_copy2.min()
    
print('the minimum is:')    
print(minimum)               
            
            
#plt.imshow(zeros.T ,origin='lower',extent=extent ,aspect='auto', cmap='coolwarm')
#plt.imshow(zeros.T ,origin='lower',extent=extent ,aspect='auto')
#plt.imshow(zeros1_copy.T ,origin='lower',extent=extent ,aspect='auto', cmap='coolwarm')
plt.imshow(zeros1_copy.T ,origin='lower',extent=extent ,aspect='auto')
plt.colorbar(label='Amount of LAP data')
plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
plt.ylabel('r (km)')

## Plot #2 Bmodulus vs Outgassing

In [ ]:
MM_prominence_data3="C:/Users/Ariel/Desktop/JUPYTER/Resultados\data_used_for_the_histograms/All_prominence_events3"

#Importing data
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmodulus', 'cometo-centric distance']
dataa= pd.read_table(str(MM_prominence_data3), header=None, names=titles, sep='\t')
data3=dataa.iloc[np.arange(1,len(dataa)),:] #delete the first row
data3=data3.reset_index(drop=True)

#print(data3)
data3_copy=data3.copy()
#Filter nan values
for j in np.arange(0,len(data3)):
    if pd.isnull(data3_copy['gas_production_rates'][j])==True or pd.isnull(data3_copy['bmodulus'][j])==True:
        data3_copy.drop(j, axis=0, inplace=True)
        
data3_copy=data3_copy.reset_index(drop=True)

#print(data3_copy)
x2=data3_copy['gas_production_rates']
xx2=x2.astype(float)
y2=data3_copy['bmodulus']
yy2=y2.astype(float)

xxx2=[]

for i in np.arange(0,len(xx2)):
    #print(i)
    if np.isnan(xx2[i])==True:
        xxx2.append(np.nan)
    
    else:
        xxx2.append(math.log(xx2[i],10))

xmin=24.5
xmax=29
ymin=0
ymax2=100        

#plt.figure()
#plt.hist2d(xxx2,yy2,bins=(100,100),range=[[xmin, xmax], [ymin, ymax2]]) #, cmap=plt.cm.jet)
#plt.colorbar(label='Amount of MM waves')
#plt.xscale('log')
#plt.ylim([min(yy),max(yy)]) #-5 and 10
#plt.ylim([0,100])
#plt.xlim([24.5,29])
#plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
#plt.ylabel('|B| (nT)')


extent=[xmin,xmax,ymin,ymax2]
HHH2, xedges, yedges= np.histogram2d(xxx2,yy2, bins=100, range=[[xmin,xmax],[ymin,ymax]])

HHH2_copy=HHH2.copy()
for i in np.arange(0,len(HHH2_copy[0])):
    for j in np.arange(0,len(HHH2_copy[0])):
        if HHH2_copy[i][j]==0:
            HHH2_copy[i][j]=np.nan


plt.figure()
#plt.imshow(HHH.T ,origin='lower', extent=extent, aspect='auto')
plt.imshow(HHH2_copy.T ,origin='lower', extent=extent, aspect='auto')
plt.colorbar(label='Amount of MM waves')
plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
plt.ylabel('|B| (nT)')




In [ ]:
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmodulus', 'cometo-centric distance']
dataa= pd.read_table(str(MM_prominence_data3), header=None, names=titles, sep='\t')
data3=dataa.iloc[np.arange(1,len(dataa)),:] #delete the first row
data3=data3.reset_index(drop=True)

print(len(data3))
print(max(data3['bmodulus'].astype(float)))
print(data3)
xx2=x2.astype(float)
plt.plot(np.arange(0,624), data3['bmodulus'].astype(float))

In [ ]:
xedges=100
yedges=100
xmin=24.5
xmax=29
ymin=0
ymax=100

#plt.plot(xxx2,yy2)
#print(xxx2)
#print(yy2)
HHH2, xedges, yedges= np.histogram2d(xxx2,yy2, bins=100, range=[[xmin,xmax],[ymin,ymax]])
H_bmodulus=matrix_division(HHH2,zeros2)

hbmodulus_copy=H_bmodulus.copy()
for i in np.arange(0,len(hbmodulus_copy[0])):
    for j in np.arange(0,len(hbmodulus_copy[0])):
        if hbmodulus_copy[i][j]==0:
            hbmodulus_copy[i][j]=np.nan

extent = [24.5,29, 0, 100]
plt.figure()
#plt.imshow(zeros2.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(H_bmodulus.T ,origin='lower', extent=extent, aspect='auto')
plt.imshow(hbmodulus_copy.T ,origin='lower', extent=extent, aspect='auto')
plt.colorbar(label='Amount of normalized MM waves')
plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
plt.ylabel('|B| (nT)')

In [ ]:
plt.figure()
extent = [24.5,29, 0, 100]

zeros2_copy=zeros2.copy()
for i in np.arange(0,len(zeros2_copy[0])):
    for j in np.arange(0,len(zeros2_copy[0])):
        if zeros2_copy[i][j]==0:
            zeros2_copy[i][j]=np.nan


plt.imshow(zeros2_copy.T ,origin='lower',extent=extent ,aspect='auto')
#plt.imshow(zeros2.T ,origin='lower',extent=extent ,aspect='auto', cmap='coolwarm')
plt.colorbar(label='Amount of LAP data')
plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
plt.ylabel('|B| (nT)')

In [ ]:
'ANALISIS DE LOS DOS DIAS'

matriz1prueba=H2[0].copy()
matriz2prueba=H2[1].copy()

print(matriz1prueba)
for i in np.arange(0,len(matriz1prueba)):
    for j in np.arange(0,len(matriz1prueba)):
        if matriz1prueba[i][j]==0:
            matriz1prueba[i][j]=np.nan
        if matriz2prueba[i][j]==0:
            matriz2prueba[i][j]=np.nan    

plt.figure()
plt.imshow(matriz1prueba.T ,origin='lower',extent=extent ,aspect='auto', cmap='coolwarm')
#plt.imshow(zeros2.T ,origin='lower',extent=extent ,aspect='auto', cmap='coolwarm')
plt.colorbar(label='Amount of LAP data')
plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
plt.ylabel('|B| (nT)')

plt.figure()
plt.imshow(matriz2prueba.T ,origin='lower',extent=extent ,aspect='auto', cmap='coolwarm')
#plt.imshow(zeros2.T ,origin='lower',extent=extent ,aspect='auto', cmap='coolwarm')
plt.colorbar(label='Amount of LAP data')
plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
plt.ylabel('|B| (nT)')

In [ ]:
zeros2 = np.zeros(shape=(100,100))    
#for j in np.arange(0,len(H2)):
for j in np.arange(0,1):    
    zeros2=np.add(zeros2,H2[j])

for i in np.arange(0,len(matriz1prueba)):
    for j in np.arange(0,len(matriz1prueba)):
        if zeros2[i][j]==0:
            zeros2[i][j]=np.nan
    
    
plt.figure()
plt.imshow(zeros2.T ,origin='lower',extent=extent ,aspect='auto', cmap='coolwarm')
#plt.imshow(zeros2.T ,origin='lower',extent=extent ,aspect='auto', cmap='coolwarm')
plt.colorbar(label='Amount of LAP data')
plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
plt.ylabel('|B| (nT)')    

In [ ]:
zeros1 = np.zeros(shape=(100,100))    
for j in np.arange(0,len(H1)):
#for j in np.arange(4,5):    
    zeros1=np.add(zeros1,H1[j])

for i in np.arange(0,len(matriz1prueba)):
    for j in np.arange(0,len(matriz1prueba)):
        if zeros2[i][j]==0:
            zeros2[i][j]=np.nan
    
    
plt.figure()
plt.imshow(zeros1.T ,origin='lower',extent=[24.5,29,0,500] ,aspect='auto', cmap='coolwarm')
#plt.imshow(zeros2.T ,origin='lower',extent=extent ,aspect='auto', cmap='coolwarm')
plt.colorbar(label='Amount of LAP data')
plt.xlabel('Q ($s^{-1}$)')#,fontsize=10)
plt.ylabel('|B| (nT)')    

In [ ]:
#VEAM0S LOS MAXIMOS DE LOS CAMPOS MAGNETICOS


#mag_folder2="C:/Users/Ariel/Desktop/JUPYTER/FINALDATAFRAMEEXAMPLENOVEMBER2014"
mag_folder2="C:/Users/Ariel/Desktop/JUPYTER/FINALDATAFRAMEEXAMPLEAPRIL2015"

#IMPORTAR LOS DATOS ACA:
directory= os.scandir(mag_folder2)
list_of_files=get_directory(mag_folder2)

newlist=[]
for item in list_of_files:
    newlist.append(mag_folder2+'/'+str(item))
#print(newlist)


mag_tables=[]
mag_tables2=[]
for j in np.arange(0,len(newlist)):
#for j in np.arange(0,2):
    #print(j)
    #Importing data
    titles=['bmodulus', 'cometo-centric distance', 'gas_production_rate',]
    data= pd.read_table(str(newlist[j]), header=None, names=titles) #, sep='\s+')
    data2=data.iloc[np.arange(1,len(data)),:] #delete the first row
    final_data=data2.reset_index(drop=True)
    
    #HISTOGRAM FOR GAS/BMODULUS
    final_dataframe_copy2=final_data.copy()
    #Filter nan values        
    final_dataframe_copy2=final_dataframe_copy2.dropna(subset=['bmodulus','gas_production_rate'])
    final_dataframe_copy2=final_dataframe_copy2.reset_index(drop=True)
    
    mag_tables.append(final_data)
    mag_tables2.append(final_dataframe_copy2)



In [ ]:
   
q=2
print(mag_tables[q]) 
print(mag_tables2[q]) 
plt.plot(np.arange(0,len(mag_tables2[q])),mag_tables2[q]['bmodulus'].astype(float))

In [ ]:
DONE

## Cometo-centric distance

In [ ]:
#THIS IS NOT NECESSARY NOW

mag_folder="C:/Users/Ariel/Desktop/ESA/Data_Rosetta/all_data"

#IMPORTAR LOS DATOS ACA:
directory= os.scandir(mag_folder)
list_of_files=get_directory(mag_folder)

newlist=[]
for item in list_of_files:
    newlist.append(mag_folder+'/'+str(item))
#print(newlist)


mag_table22=[]
#for j in np.arange(0,len(newlist)):
for j in np.arange(0,2):
    print(j)
    #Importing data
    titles=['Dates_and_Hours', '?', 'x', 'y', 'z', 'Bx', 'By', 'Bz', 'flag']
    data= pd.read_table(str(newlist[j]), header=None, names=titles, sep='\s+', parse_dates=['Dates_and_Hours'])
    #----------------------------------------
    #Filling the data gaps
    data.Dates_and_Hours = data.Dates_and_Hours.dt.round('1s') #round to one second for simplicity
    if data.shape[0] < 86400: # if the number of datapoints is lower than one day:
        #print('Data gaps detected, padding array....')
        data2 = data.rename(index=(data['Dates_and_Hours']-data.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
        data2 = data2.reindex(range(0,86400)) # now we just fill in the missing values
        newt = pd.date_range(start = data.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
        data2['Dates_and_Hours'] = newt # now fill in the times so there is no NaT
        data = data2 
        del(data2)
    elif data.shape[0] > 86400:
        error('Data file is too long, probably need to debug the code again....')
    
    XX = data[['Dates_and_Hours','x','y','z','Bx', 'By', 'Bz']]
    
    #Calculates the magnetic field modulus for each point
    bmodulus=[]
    for i in np.arange(0,len(XX)):
        Bpoint=(data['Bx'][i]**2+data['By'][i]**2+data['Bz'][i]**2)**(1/2)
        bmodulus.append(Bpoint)
        
    #Transform a list into a data column
    df1 = pd.DataFrame({'col':bmodulus})
    XX['bmodulus']=df1
    
    #Calculates the cometo-distance for each point
    distance=[]
    for i in np.arange(0,len(XX)):
        dpoint=(data['x'][i]**2+data['y'][i]**2+data['z'][i]**2)**(1/2)
        distance.append(dpoint)
        
    #Transform a list into a data column
    df2 = pd.DataFrame({'col':distance})
    XX['cometo-centric distance']=df2
    mag_table22.append(XX)


In [ ]:
print(len(mag_table22))

In [ ]:
MM_prominence_data2="C:/Users/Ariel/Desktop/JUPYTER/histogram/All_prominence_events2"

#Importing data
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates']
data= pd.read_table(str(MM_prominence_data2), header=None, names=titles, sep='\t')
data2=data.iloc[np.arange(1,len(data)),:] #delete the first row
data2=data2.reset_index(drop=True)

path_time= pd.to_datetime(data2['Beginning']) 
path_time2= pd.to_datetime(data2['End']) 
q=path_time.dt.year
qq=path_time.dt.month
qqq=path_time.dt.day


bmodulus_list=[]
distance_list=[]

for i in np.arange(0,len(data2)):
    print(i)
    year=q[i]
    month=qq[i]
    day=qqq[i]
    
    for j in np.arange(0,len(mag_table22)):
     
        if pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.year[0]==year and pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.month[0]==month and pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.day[0]==day:   #
            day_chosen=mag_table22[j]         
            
            # Chosing the specific time interval
            filtered_df = day_chosen.loc[(day_chosen['Dates_and_Hours'] >= path_time[i])  & (day_chosen['Dates_and_Hours'] <= path_time2[i])]
            filtered_df=filtered_df.reset_index(drop=True)
        
            # find the max value of deltaB
            listA=filtered_df['deltas'].tolist()
            max_val = max(listA)
            # Its index
            index=listA.index(max_val)
            
            bmodulus_list.append(filtered_df['bmodulus'][index])
            distance_list.append(filtered_df['cometo-centric distance'][index])  
            break
    
   


    

In [ ]:
sas = pd.DataFrame({'bmodulus':bmodulus_list})
sas2 = pd.DataFrame({'cometo-centric distance':distance_list})
#print(sas)
#print(sas2)
#print(data2)

data2_copy=data2.copy()

data2_copy['bmodulus']=sas
data2_copy['cometo-centric distance']=sas2
print(data2_copy)

import csv

data2_copy.to_csv('All_prominence_events3', index=False, sep='\t')

## Heliocentric distance and Gas production rates

In [ ]:
outgassing="C:/Users/Ariel/Desktop/ESA/Data_GasProduction/gasproduction.txt"

title2=['Timestamp','Neutral gas density','Derived gas production rate']
path= str(outgassing)
data1= pd.read_table(path, header=None, names=title2, sep='\s+', engine='python')
 
#Filtering
outgassing_copy=data1.copy() 
for j in np.arange(0,len(outgassing_copy)):
    if outgassing_copy['Derived gas production rate'][j]<1000000: 
        outgassing_copy['Timestamp'][j]=np.nan
        
for j in np.arange(0,len(outgassing_copy)):
    if pd.isnull(outgassing_copy['Timestamp'][j])==True:
        outgassing_copy.drop(j, axis=0, inplace=True)
        
        
data_gas=outgassing_copy.reset_index(drop=True)

print(data_gas)

In [ ]:
print(getting_day3(data_gas,2014,12,24,1,4)[0])
print('-----')
print(getting_day3(data_gas,2014,12,24,1,4)[1])

In [ ]:
all_events="C:/Users/Ariel/Desktop/JUPYTER/histogram/All_prominence_events"
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson']
data= pd.read_table(str(all_events), header=None, names=titles, sep='\t')
data2=data.iloc[np.arange(1,len(data)),:] #delete the first row
final_data=data2.reset_index(drop=True)

print(final_data.iloc[np.arange(0,8),:])

path_time= pd.to_datetime(data2['Beginning'])
q=path_time.dt.year
qq=path_time.dt.month
qqq=path_time.dt.day
qqqq=path_time.dt.hour
qqqqq=path_time.dt.minute

gas_production_list=[]
for i in np.arange(1,len(final_data)+1):
    #It is needed to obtain the year, month and day of an specific path
    year=q[i]
    month=qq[i]
    day=qqq[i]
    hour=qqqq[i]
    minute=qqqqq[i]
    gas_production_list.append(float(getting_day3(data_gas,year,month,day,hour,minute)[1]))



In [ ]:
#print(gas_production_list)
ee=gas_production_list[622]
gas_production_list2=gas_production_list.copy()
gas_production_list2.append(ee)
print(len(gas_production_list))
#print(gas_production_list2)


In [ ]:
ses = pd.DataFrame({'gas_production_rate':gas_production_list2})
#print(ses)
#print(final_data)

final_data['gas_production_rate']=ses
#print(final_data)

import csv

#final_data.to_csv('All_prominence_events2', index=False, sep='\t')

In [ ]:
print(H_list)

In [ ]:
with np.printoptions(threshold=np.inf):
    print(data2)
    
import csv

data2.to_csv('snn', index=False, sep='\t')

In [ ]:
sis=data2.copy()

print(sis)

pss=sis.dropna(subset=['bmodulus','gas_production_rates'])
print(pss)

## NOW WE WANT THE MEAN VALUE OF THE BMODULUS IN THE FINAL LIST OF MIRROR MODE WAVES

In [ ]:
mag_table22=mag_table.copy()

In [ ]:
MM_prominence_data2="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events2"

#Importing data
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates']
data= pd.read_table(str(MM_prominence_data2), header=None, names=titles, sep='\t')
data2=data.iloc[np.arange(1,len(data)),:] #delete the first row
data2=data2.reset_index(drop=True)
print('data2')
print(data2)
path_time= pd.to_datetime(data2['Beginning']) 
path_time2= pd.to_datetime(data2['End']) 
q=path_time.dt.year
qq=path_time.dt.month
qqq=path_time.dt.day


bmodulus_list=[]
distance_list=[]

for i in np.arange(0,len(data2)):
#for i in np.arange(1,2):
    print(i)
    year=q[i]
    month=qq[i]
    day=qqq[i]
    #print(year)
    #print(month)
    #print(day)
    for j in np.arange(0,len(mag_table22)):
     
        if pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.year[0]==year and pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.month[0]==month and pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.day[0]==day:   #
            day_chosen=mag_table22[j]         
            
            # Chosing the specific time interval
            filtered_df = day_chosen.loc[(day_chosen['Dates_and_Hours'] >= path_time[i])  & (day_chosen['Dates_and_Hours'] <= path_time2[i])]
            filtered_df=filtered_df.reset_index(drop=True)
            #print('filtered')
            #print(filtered_df)
            # find the mean bmodulus value of the interval
            listA=filtered_df['bmodulus'].tolist()
            #max_val = max(listA)
            mean_val=np.mean(listA)
            #print('mean_val')
            #print(mean_val)
            'Its index'
            #index=listA.index(mean_val)
            #bmodulus_list.append(filtered_df['bmodulus'][index])
            bmodulus_list.append(mean_val)
            
            'The variation of the cometo-centric distance is negligible in an interval of 60 seconds aproximately'
            #distance_list.append(filtered_df['cometo-centric distance'][index])  
            break

In [ ]:
print(len(bmodulus_list))
plt.plot(np.arange(0,len(bmodulus_list)), bmodulus_list)

In [ ]:
data3=data2.copy()
data3['bmodulusaverage']=bmodulus_list
import csv
data3.to_csv('finaldataframe4_', index=False, sep='\t')  


## NOW WE WANT THE x,y,z and z/y values  IN THE FINAL LIST OF MIRROR MODE WAVES

In [ ]:
mag_table22=mag_table.copy()

In [ ]:
MM_prominence_data2="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events8"

#Importing data
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates','bmoduluspeaks','cometo-centric distance','x','roots','bmodulusaverage']
data= pd.read_table(str(MM_prominence_data2), header=None, names=titles, sep='\t')
data2=data.iloc[np.arange(1,len(data)),:] #delete the first row
data2=data2.reset_index(drop=True)
print('data2')
print(data2)
path_time= pd.to_datetime(data2['Beginning']) 
path_time2= pd.to_datetime(data2['End']) 
q=path_time.dt.year
qq=path_time.dt.month
qqq=path_time.dt.day


x_list=[]
y_list=[]
z_list=[]
for i in np.arange(0,len(data2)):
#for i in np.arange(0,1):
    print(i)
    year=q[i]
    month=qq[i]
    day=qqq[i]
    #print(year)
    #print(month)
    #print(day)
    for j in np.arange(0,len(mag_table22)):
     
        if pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.year[0]==year and pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.month[0]==month and pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.day[0]==day:   #
            day_chosen=mag_table22[j]         
            
            # Chosing the specific time interval
            filtered_df = day_chosen.loc[(day_chosen['Dates_and_Hours'] >= path_time[i])  & (day_chosen['Dates_and_Hours'] <= path_time2[i])]
            filtered_df=filtered_df.reset_index(drop=True)
            #print('filtered')
            #print(filtered_df)
            
            
            # find the mean bmodulus value of the interval
            listA=filtered_df['x'].tolist()
            mean_valA=np.mean(listA)
            listB=filtered_df['y'].tolist()
            mean_valB=np.mean(listB)
            listC=filtered_df['z'].tolist()
            mean_valC=np.mean(listC)
            
            #print('means')
            #print(mean_valA)
            #print(mean_valB)
            #print(mean_valC)
         
            x_list.append(mean_valA)
            y_list.append(mean_valB)
            z_list.append(mean_valC)
              
            break

In [ ]:
data3=data2.copy()
print(data3)
data3['x']=x_list
data3['y']=y_list
data3['z']=z_list

import csv
print(data3)
data3.to_csv('All_prominence_events9', index=False, sep='\t')
